# <span style='color:red'>Project 1: Dynamic Programming for Trade Scheduling</span>

In this project, we use dynamic programming to create a trading schedule that maximizes the total number of shares traded under a model of liquidity impact with memory.

The project has two computational goals:

1. first, develop a scalable **approximate** solution for the large problem;
2. then, solve the **same original integer-valued problem exactly** and compare the two approaches.


In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width: 90% !important; }</style>"))

import csv
import sys
import numpy as np
import math
import matplotlib.pyplot as plt
import pandas as pd
import time


## Model

Suppose we have a total of $N$ shares that we would like to **attempt to trade** over $T$ time periods. We choose an order schedule
$$
(n_0,n_1,\ldots,n_{T-1}), \qquad n_i\in\mathbb Z_{\ge 0},
$$
subject to
$$
\sum_{i=0}^{T-1} n_i=N.
$$
Equivalently, in the final period we submit all shares that have not yet been scheduled:
$$
n_{T-1}=N-\sum_{i=0}^{T-2}n_i.
$$

The market does not necessarily allow the full submitted quantity to trade. We consider the following model. Let $M_{-1}=0$. Given parameters $\alpha>0$ and $0<\pi<1$, the process evolves as follows for $i=0,1,\ldots,T-1$:

1. After submitting $n_i$ shares, update the liquidity-memory
$$
M_i=\left\lceil \frac{M_{i-1}+n_i}{2}\right\rceil.
$$

2. The number of shares actually traded under this model in period $i$ is
$$
s_i=
\left\lceil
\bigl[1-\alpha M_i^\pi\bigr]_+\,n_i
\right\rceil,
\qquad [x]_+:=\max\{x,0\}.
$$

The objective is to choose the order schedule $(n_0,\ldots,n_{T-1})$ to maximize
$$
\sum_{i=0}^{T-1}s_i.
$$

**Notation:** $n_i$ denotes the submitted order size; $s_i$ denotes the model-implied traded quantity. We keep these two quantities distinct throughout the project.


### Example

Take
$$
N=10{,}000,\qquad T=4,\qquad \alpha=0.001,\qquad \pi=0.5,
$$
and consider the order schedule
$$
(5000,1000,2000,2000).
$$


In [2]:
def simulate_schedule(schedule, alpha, pi):
    """Simulate the liquidity-memory model for a given integer order schedule."""
    M = 0
    fills = []
    memory = []

    for n_i in schedule:
        n_i = int(n_i)
        M = math.ceil(0.5 * (M + n_i))
        traded = math.ceil(max(0.0, 1.0 - alpha * M**pi) * n_i)

        memory.append(M)
        fills.append(traded)

    return np.asarray(fills, dtype=int), np.asarray(memory, dtype=int)

N = 10_000
T = 4
alpha = 1e-3
pi = 0.5
schedule = np.array([5000, 1000, 2000, 2000], dtype=int)

assert schedule.sum() == N
fills, memory = simulate_schedule(schedule, alpha, pi)

for i in range(T):
    print(f"period {i}: order = {schedule[i]:5d}, M = {memory[i]:5d}, traded = {fills[i]:5d}")
print("total traded =", fills.sum())


period 0: order =  5000, M =  2500, traded =  4750
period 1: order =  1000, M =  1750, traded =   959
period 2: order =  2000, M =  1875, traded =  1914
period 3: order =  2000, M =  1938, traded =  1912
total traded = 9535


## <span style='color:red'>Task 1: Dynamic programming formulation and implementation</span>

Design a dynamic-programming algorithm that computes the optimal order schedule
$$
(n_0^*,n_1^*,\ldots,n_{T-1}^*)
$$
for the model above.

Before implementing your algorithm, clearly identify:

- the **state**;
- the **action** and feasible-action set;
- the **state transition**;
- the **one-period reward**;
- the **last-period condition**;
- the **Bellman recursion**.


## Task 1(a): Large-scale approximate DP

First, solve the large problem allowing numerical approximation.

You may use state-space discretization, interpolation, or other reasonable numerical approximations. You are also encouraged to use computational techniques such as vectorization and precomputation.

Your implementation should be designed to run in a reasonable amount of time for $N=100000$ and $T=10$.

**Requirements**

1. Clearly describe every approximation you use.
2. Report the approximate optimal schedule and the corresponding total number of shares traded.
3. Report the runtime of your implementation.
4. Perform at least one **refinement check**: make the approximation finer and report how the objective value and/or schedule changes.

There is no single required approximation method.

## Task 1(b): Exact DP

Now solve the **same problem exactly**.

For this part:

- do **not** discretize the state space;
- do **not** interpolate the value function;
- do **not** restrict the action space heuristically;
- your algorithm must optimize over the original integer-valued problem.

Your code should still run for
$$
N=100000,\qquad T=10,\qquad \alpha=0.001,\qquad \pi=0.5.
$$

Report:

1. the exact optimal schedule $(n_0^*,\ldots,n_9^*)$;
2. the exact optimal total number of shares traded;
3. runtime;
4. a measure of computational size, such as the number of states stored/processed at each period;
5. a comparison with your approximate solution from Task 1(a).

### Small hint
A dense table over every theoretically possible state is not necessarily the best representation.  
Think about whether a **forward DP** can keep only states that are actually reached as the algorithm progresses.






### Validation

Validate your exact implementation on at least one much smaller instance for which you can independently check the answer (for example, by brute-force enumeration or naive DP implementation).

Your exact DP and the independent calculation should return the same optimal objective value.

# <span style='color:red'>Task 2: Evaluate the order schedule using TSLA volume data</span>

Use the **exact optimal schedule from Task 1(b)**,
$$
(n_0^*,n_1^*,\ldots,n_9^*).
$$



The file `TSLA.csv` contains 1-minute TSLA data. For each trading day, use the first two hours of regular trading, from 9:30 a.m. through 11:29 a.m. (Use the Volume column from the Trade block of the CSV, not the Bid or Ask blocks.)

Divide these 120 minutes into 12 consecutive 10-minute intervals.

For one 10-minute interval, let
$$
(V_0,V_1,\ldots,V_9)
$$
be the TSLA traded volumes in its ten one-minute bars.

The interval score is computed using:
$$
\boxed{
\text{Interval Score}
=
\sum_{i=0}^{9}\min\{n_i^*,V_i\}.
}
$$

The **TOTAL SCORE** is the average of the interval scores over the first 12 intervals of every day in the **first half of the dataset**.

For example, if the data contain 300 trading days, use the first 150 days and therefore $12\times150=1800$ intervals.

Using the schedule from Task 1(b):

1. Compute its TOTAL SCORE on the first half of the data.
2. Compare it with the TOTAL SCORE of the equal-sized schedule
$$
(N/T,\ldots,N/T).
$$







## <span style='color:red'>Task 3: Calibrate $\pi$ and evaluate out of sample</span>

Keep
$$
N=100{,}000,\qquad T=10,\qquad \alpha=0.001.
$$

Use the **first half of the data only** to choose $\pi$.

1. Search over values of $\pi$ in the interval
$$
[0.3,0.7].
$$
You may use a sufficiently fine grid or a coarse-to-fine search. State the values you evaluate.

2. For each candidate $\pi$:
   - use the exact DP algorithm from Task 1(b) to obtain an order schedule $n^*(\pi)$;
   - compute the training TOTAL SCORE of that schedule using Task 2.

3. Let $\pi^*$ be the value giving the highest training TOTAL SCORE. Report $\pi^*$, the corresponding schedule, and its training score. Plot TOTAL SCORE versus $\pi$ for the values you evaluated.

4. **Out-of-sample evaluation:** without changing $\pi^*$, compute the TOTAL SCORE of $n^*(\pi^*)$ on the second half of the trading days. Compare the training and out-of-sample scores.

